# 8차 제출 파일 만들기: 8차 최종 방법으로 predict_input 927행 예측

| 단계 | 내용 |
|---|---|
| 1 | 8차 최종 방법(`results/final_method.json`): 모델 목록 · 스태킹 계수 · 임계값 |
| 2 | 테스트 세트 1회 확인: 구성 모델을 개발 6,400행으로 학습 → 테스트 1,600행 예측 → 결합 |
| 3 | 8,000행 전체로 재학습 → predict_input 예측 → `predict_output_임채현_8차.csv` |

In [1]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
HERE = Path.cwd(); M = HERE.parent
for p in ("common", "exp03_tuning", "exp06_knn_features"): sys.path.insert(0, str(M / p))
from preprocess import preprocess_train, preprocess_predict, load_feature_columns
from evaluate import load_splits, predict_score
from submission import write_submission, test_report, INPUT
from load_features import load_all, load_predict, blocks_for
from knnfeat import full_fp
import registry_ext as RX

STAGE = "8차"
final = json.load(open(HERE / "results" / "final_method.json"))
print(final["method"], "| 모델 수", len(final["models"]), "| 임계값", final["threshold"], "| CV F1", round(final["cv_f1"], 4))
df, X, y, g = preprocess_train(M.parent / "train.csv"); df = load_splits(df); cols = load_feature_columns()
ALL = pd.concat([load_all(df, X), full_fp(df).reset_index(drop=True)], axis=1); dev = (df.set == "dev").values
specs = {n: RX.build(n, cols) for n in final["models"]}
need = sorted({c for _, use in specs.values() for c in use if not c.startswith("full")})

def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def combine(P):
    if final["kind"] == "mean":
        return np.sum([w * P[n] for n, w in zip(final["models"], final["weights"])], 0)
    z = final["intercept"] + np.sum([w * logit(P[n]) for n, w in zip(final["models"], final["weights"])], 0)
    return 1 / (1 + np.exp(-z))
def predict_all(tr_mask, X_te):
    P = {}
    for n, (mk, use) in specs.items():
        t0 = time.time(); m = mk(); m.fit(ALL.loc[tr_mask, use], y[tr_mask]); P[n] = predict_score(m, X_te[use])
    return P

지금까지 최고 + 8차 2개 | 모델 수 41 | 임계값 0.42 | CV F1 0.8309


## 테스트 세트 1회 확인

In [2]:
t0 = time.time()
P = predict_all(dev, ALL.loc[~dev])
rep = test_report(y[~dev], combine(P), final["threshold"])
rep05 = test_report(y[~dev], combine(P), 0.5)
print("최종 임계값", {k: round(v, 4) for k, v in rep.items()}, f"| {time.time()-t0:.0f}s")
print("임계값 0.5 ", {k: round(v, 4) for k, v in rep05.items()})
pd.DataFrame([rep, rep05], index=[final["method"], final["method"] + " (임계값 0.5)"]).to_csv(HERE / "results" / "test_score.csv")

최종 임계값 {'F1': 0.8254, 'AUC': 0.8793, 'Precision': 0.7817, 'Recall': 0.8744, 'threshold': 0.42} | 967s
임계값 0.5  {'F1': 0.8207, 'AUC': 0.8793, 'Precision': 0.8013, 'Recall': 0.841, 'threshold': 0.5}


## 8,000행 전체로 재학습 후 예측

In [3]:
pdf, Xp = preprocess_predict(INPUT)
XP = pd.concat([load_predict(pdf, Xp, blocks_for(need)), full_fp(pdf).reset_index(drop=True)], axis=1)
P = predict_all(np.ones(len(df), bool), XP)
p = combine(P)
path, sub = write_submission(pdf["SMILES"], (p >= final["threshold"]).astype(int), HERE, STAGE)
print("저장:", path.name, "| 행 수", len(sub), "| 예측 비독성(1) 비율", round(sub.iloc[:, 1].mean(), 3), "(train 0.542)")
pd.DataFrame({"SMILES": pdf["SMILES"], "score_nontoxic": p}).to_csv(HERE / "results" / "predict_proba.csv", index=False)
path05, sub05 = write_submission(pdf["SMILES"], (p >= 0.5).astype(int), HERE, STAGE + "_thr0.5")
print("임계값 0.5 버전:", path05.name, "| 예측 비독성(1) 비율", round(sub05.iloc[:, 1].mean(), 3))
for k, f in (("3차", "exp03_tuning/predict_output_임채현_3차.csv"), ("4차", "exp04_threshold_ensemble/predict_output_임채현_4차.csv")):
    prev = pd.read_csv(M / f).iloc[:, 1].values
    for lab, s_ in (("최종 임계값", sub), ("임계값 0.5", sub05)):
        v = s_.iloc[:, 1].values
        print(f"{k} 제출과 다른 행 ({lab}): {int((prev != v).sum())} (우리만 1: {int(((v == 1) & (prev == 0)).sum())}, {k}만 1: {int(((v == 0) & (prev == 1)).sum())})")
P_test = None
display(sub.head())

저장: predict_output_임채현_8차.csv | 행 수 927 | 예측 비독성(1) 비율 0.595 (train 0.542)
임계값 0.5 버전: predict_output_임채현_8차_thr0.5.csv | 예측 비독성(1) 비율 0.561
3차 제출과 다른 행 (최종 임계값): 48 (우리만 1: 42, 3차만 1: 6)
3차 제출과 다른 행 (임계값 0.5): 38 (우리만 1: 21, 3차만 1: 17)
4차 제출과 다른 행 (최종 임계값): 25 (우리만 1: 11, 4차만 1: 14)
4차 제출과 다른 행 (임계값 0.5): 37 (우리만 1: 1, 4차만 1: 36)


,SMILES,output
0,OC(=O)c1cc2sccc2[nH]1,1
1,[O-][n+]1onc(c2ccccc2)c1C#N,1
2,CN1C(=O)N(C)c2ncn(C)c2C1=O,1
3,Clc1cccc(c1)C2CNCC=CC2,1
4,CCN(CC)CC(=O)Nc1c(C)cccc1C,1
